## 2026-06 Rectangular Gradient Widget
This widget creates a grid that generates a gradient from four anchor points. Each anchor point represents the corner of a rectangle. To fill in the rectangle smoothly cells generated within the grid have their hue, saturation, and lightness calculated based off of the influence (weights) of the four corners.

Knowns:
- points are tuples point = (hue, saturation, lightness)
- hue is angular and loops around a circle 0 to 360
- this gets converted to rgb to enable weighting logic
- we have 4 anchor points that represent TL, TR, BL, BR
- we can determine grid dimensions (ex.5x3)
- we can define unit_size (ex. 5px)

Unknowns:
- The values of each unit between the anchor points

Interaction:
- we can generate an image that shows the grid
- we can copy any unit hsl value to the clipboard
- any unit value we click can output it's hsl value

Dependencies:
- To convert HSL(HLS) to RGB we are using Python's colorsys.() conversion module
- We are using anywidgets, traitlets, and ipywidgets to bridge JS and Py in Jupyter

### Why HSL?
This is simply personal preference. I think in terms of HSL and have found it easier in a lot of cases to build color themes in browser using HSL. However, you can remove the conversion within the widget that goes from RGB to HSL and replace it with the color framework of your choosing.

## 01 The Widget Class

In [5]:
import anywidget
import traitlets
import ipywidgets as widgets

class GradientWidget(anywidget.AnyWidget):
    _esm = """
    function render({model, el}) {
        const data = model.get('grid_data');
        const unitSize = model.get('unit_size');
        const cols = model.get('cols');

        const grid = document.createElement('div');
        grid.style.display = 'inline-grid';
        grid.style.gridTemplateColumns = `repeat(${cols}, ${unitSize}px)`;
        grid.style.gap = '0';

        const info = document.createElement('div');
        info.style.cssText = 'margin-top: 6px; font: 12px monospace; color: #555;';
        info.textContent = 'Click a cell to see its HSL values';

        data.forEach(([h, s, l]) => {
            const cell = document.createElement('div');
            cell.style.width = unitSize + 'px';
            cell.style.height = unitSize + 'px';
            cell.style.background = `hsl(${h}, ${s}%, ${l}%)`;
            cell.style.border = 'solid 1px #fff';

            cell.addEventListener('click', () => {
                const text = `hsl(${Math.round(h)}, ${Math.round(s)}%, ${Math.round(l)}%)`;
                navigator.clipboard.writeText(text)
                    .then(() => info.textContent = 'Copied ' + text)
                    .catch(() => info.textContent = text);
                model.set('selected_hsl', text);
                model.save_changes();
            });

            grid.appendChild(cell);
        });

        el.appendChild(grid);
        el.appendChild(info);
    }
    export default { render };
    """
    grid_data    = traitlets.List([]).tag(sync=True)
    unit_size    = traitlets.Int(20).tag(sync=True)
    cols         = traitlets.Int(4).tag(sync=True)
    rows         = traitlets.Int(2).tag(sync=True)
    selected_hsl = traitlets.Unicode("").tag(sync=True)

## 02 The Main Function

In [6]:
import colorsys

def gradient_widget (TL, TR, BL, BR, dimensions, unit_size=30) :
    cols, rows = dimensions
    def calc_grid (TL, TR, BL, BR, dimensions) : 
        width, height = dimensions
        
        def calc_cell (row, col, dimensions, TL, TR, BL, BR) :
            u = col / (width - 1)
            v = row / (height - 1)
            H_TL, S_TL, L_TL = TL
            H_TR, S_TR, L_TR = TR
            H_BL, S_BL, L_BL = BL
            H_BR, S_BR, L_BR = BR

            rgb_TL = colorsys.hls_to_rgb((H_TL / 360), (L_TL / 100), (S_TL / 100))
            rgb_TR = colorsys.hls_to_rgb((H_TR / 360), (L_TR / 100), (S_TR / 100))
            rgb_BL = colorsys.hls_to_rgb((H_BL / 360), (L_BL / 100), (S_BL / 100))
            rgb_BR = colorsys.hls_to_rgb((H_BR / 360), (L_BR / 100), (S_BR / 100))
        
            w_TL = (1-u) * (1-v)    # this weight is largest at (0,0)
            w_TR = u * (1-v)     # largest weight at (1,0)
            w_BL = (1-u) * v     # largest at (0,1)
            w_BR = u * v     # largest at (1,1)
        
            # To calculate we can apply weights directly
            R = (w_TL * rgb_TL[0]) + (w_TR * rgb_TR[0]) + (w_BL * rgb_BL[0]) + (w_BR * rgb_BR[0])
            G = (w_TL * rgb_TL[1]) + (w_TR * rgb_TR[1]) + (w_BL * rgb_BL[1]) + (w_BR * rgb_BR[1])
            B = (w_TL * rgb_TL[2]) + (w_TR * rgb_TR[2]) + (w_BL * rgb_BL[2]) + (w_BR * rgb_BR[2])

            h, l, s = colorsys.rgb_to_hls(R, G, B)
            H = h* 360
            S = s * 100
            L = l * 100
        
            return H, S, L
    
        grid = []
        for r in range(height) :
            current_row = []
            for c in range(width):
                cell = calc_cell(r, c, dimensions, TL, TR, BL, BR)
                current_row.append(cell)
            grid.append(current_row)
        return grid

    grid = calc_grid(TL, TR, BL, BR, dimensions)


    flat = [[round(h, 1), round(s, 1), round(l, 1)]
        for row in grid for h, s, l in row]

    w = GradientWidget(
        cols=cols,
        rows=rows,
        unit_size=unit_size,
        grid_data=flat,
    )

    out = widgets.Output()

    def on_click(change):
        with out:
            print(change['new'])

    w.observe(on_click, names=['selected_hsl'])
    return widgets.VBox([w, out])

## 03 The Widget

In [7]:
# Day
gradient_widget(
    TL=(18, 85, 63),
    TR=(45, 75, 72),
    BL=(342, 85, 63),
    BR=(216, 75, 100),
    dimensions=(9, 9),
)

In [8]:
# Night
gradient_widget(
    TL=(18, 75, 100),
    TR=(225, 75, 85),
    BL=(342, 75, 54),
    BR=(216, 75, 27),
    dimensions=(9, 9),
)